# VAE's decoder architecture

In [ ]:
import torch as th
import numpy as np
from einops import rearrange
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
from importlib import reload
import vae_decoder
from vae_decoder import unpatchify, Head
reload(vae_decoder)

# Dev
device = 'cuda'
J = 25
out_J_chn = 2
# 0 resblock = we have at least 1 resblock in the decoder (during upsampling) see vae_decoder.py
vae_decoder = vae_decoder.JointVAE38(J=J, out_J_chn=out_J_chn, z_dim=48, num_res_blocks=0).to(device)
print(vae_decoder)
for name, param in vae_decoder.named_parameters():
    print(name, param.shape, param.requires_grad)

JointVAE38(
  (model): VAEDecoder38(
    (conv2): CausalConv3d(48, 48, kernel_size=(1, 1, 1), stride=(1, 1, 1))
    (decoder): Decoder3d_38(
      (conv1): CausalConv3d(48, 1024, kernel_size=(3, 3, 3), stride=(1, 1, 1))
      (middle): Sequential(
        (0): ResidualBlock(
          (residual): Sequential(
            (0): RMS_norm()
            (1): SiLU()
            (2): CausalConv3d(1024, 1024, kernel_size=(3, 3, 3), stride=(1, 1, 1))
            (3): RMS_norm()
            (4): SiLU()
            (5): Dropout(p=0.0, inplace=False)
            (6): CausalConv3d(1024, 1024, kernel_size=(3, 3, 3), stride=(1, 1, 1))
          )
          (shortcut): Identity()
        )
        (1): AttentionBlock(
          (norm): RMS_norm()
          (to_qkv): Conv2d(1024, 3072, kernel_size=(1, 1), stride=(1, 1))
          (proj): Conv2d(1024, 1024, kernel_size=(1, 1), stride=(1, 1))
        )
        (2): ResidualBlock(
          (residual): Sequential(
            (0): RMS_norm()
            (1

In [ ]:
# data = th.load('/host/data2/mint/Motion_Dataset/SkelAg/frontview_bodyjoints_320p/dit_feats/0/0.pth', weights_only=False, map_location='cpu')
# data = th.load('/host/data2/mint/Motion_Dataset/SkelAg/frontview_bodyjoints_480p/dit_feats/0/0.pth', weights_only=False, map_location='cpu')
data = th.load('/host/data2/mint/Motion_Dataset/SkelAg/frontview_bodyjoints_320p_45frames/dit_feats/0/0.pth', weights_only=False, map_location='cpu')
dit_features = data[0]['dit_features']
grid_size = data[0]['grid_size']
patch_size = data[0]['patch_size']
dim = data[0]['dim']
out_dim = data[0]['out_dim']
tiled = data[0]['tiled']
tile_size = data[0]['tile_size']
tile_stride = data[0]['tile_stride']
z_dim = data[0]['z_dim']  # vae z_dim
print("num frames:", data[0]['num_frames'])
print("joints_3d:", data[0]['joints_3d'].shape)
print("joints_2d:", data[0]['joints_2d'].shape)
print("patch_size:", patch_size)
print("dim:", dim)
print("out_dim:", out_dim)
print("z_dim:", z_dim)
print("tiled:", tiled)
print("tile_size:", tile_size)
print("tile_stride:", tile_stride)
print(data[0].keys())
print(dit_features.shape, grid_size)
print(dit_features[0:1].shape, grid_size[0], grid_size[1], grid_size[2])

num frames: 45
joints_3d: (45, 25, 3)
joints_2d: (45, 25, 3)
patch_size: [1, 2, 2]
dim: 3072
out_dim: 48
z_dim: 48
tiled: True
tile_size: (30, 52)
tile_stride: (15, 26)
dict_keys(['input_video', 'height', 'width', 'num_frames', 'cfg_scale', 'tiled', 'tile_size', 'tile_stride', 'rand_device', 'use_gradient_checkpointing', 'use_gradient_checkpointing_offload', 'cfg_merge', 'vace_scale', 'max_timestep_boundary', 'min_timestep_boundary', 'preferred_timestep_id', 'preferred_dit_block_id', 'joints_3d', 'joints_2d', 'cams_intr', 'cams_extr', 'joint_names', 'bones', 'input_image', 'noise', 'latents', 'input_latents', 'fuse_vae_embedding_in_latents', 'first_frame_latents', 'vace_context', 'animate_pose_video', 'animate_face_video', 'animate_inpaint_video', 'animate_mask_video', 'prompt', 'context', 'dit_features', 'grid_size', 'dim', 'out_dim', 'patch_size', 'z_dim'])
torch.Size([30, 1, 2400, 3072]) (12, 10, 20)
torch.Size([1, 1, 2400, 3072]) 12 10 20


In [ ]:
def unpatchify(x, grid_size):
    patch_size = [1, 2, 2]
    return rearrange(
        x, 'b (f h w) (x y z c) -> b c (f x) (h y) (w z)',
        f=grid_size[0], h=grid_size[1], w=grid_size[2], 
        x=patch_size[0], y=patch_size[1], z=patch_size[2]
    )
    
# 3072 48 [1, 2, 2] 1e-06
dim = 3072
out_dim = 48
patch_size = [1, 2, 2]
eps = 1e-6

inp = dit_features[0].type(th.float32).to(device)
head = Head(dim=dim, out_dim=out_dim, patch_size=patch_size, eps=eps).to(device)
print("Input: ", inp.shape)
out_head = head(inp)
print("Out head: ", out_head.shape)
out_unpatched = unpatchify(out_head, grid_size)
print("Out unpatched: ", out_unpatched.shape)
# assert False
out_tiled = vae_decoder.decode(out_unpatched[:, :, :4, :, :], device=device, tiled=tiled, tile_size=tile_size, tile_stride=tile_stride).cpu()
print(out_tiled.shape)
# out_notiled = vae_decoder.decode(out_unpatched[:, :, :2, :, :], device=device).cpu()
# print(out_notiled.shape)
# out_diff = th.abs(out_tiled - out_notiled)
# print("Diff tiled vs notiled: ", out_diff.max(), out_diff.mean())

# Imitate training loops

In [ ]:
import tqdm
T = 2
H = 320
W = 640
eps = 1e-6
gt = th.randn(1, 3, 4 * T + 1, H, W)
inp = dit_features[0].type(th.float32).to(device)

def unpatchify(x, grid_size):
    patch_size = [1, 2, 2]
    return rearrange(
        x, 'b (f h w) (x y z c) -> b c (f x) (h y) (w z)',
        f=grid_size[0], h=grid_size[1], w=grid_size[2], 
        x=patch_size[0], y=patch_size[1], z=patch_size[2]
    )

head = Head(dim=dim, out_dim=out_dim, patch_size=patch_size, eps=eps).to(device)
optimizer = th.optim.Adam(list(head.parameters()) + list(vae_decoder.parameters()), lr=1e-4)
    
for name, param in head.named_parameters():
    print(name, param.shape, param.requires_grad)
for name, param in vae_decoder.named_parameters():
    param.requires_grad = True
    print(name, param.shape, param.requires_grad)

loss_list = []
t = tqdm.trange(100)
for i in t:
    vae_decoder.model.clear_cache()
    optimizer.zero_grad()
    # print(inp.shape)
    out_head = head(inp)
    # print(out_head.shape)
    out_unpatched = unpatchify(out_head, grid_size)
    # print(out_unpatched.shape)
    # out_tiled = vae_decoder.decode(out_unpatched[:, :, :T+1, :, :], device=device, tiled=tiled, tile_size=tile_size, tile_stride=tile_stride)
    out_tiled = vae_decoder.decode(out_unpatched[:, :, :T+1, :, :], device=device)
    loss = th.nn.functional.mse_loss(out_tiled.to(device), gt.to(device))
    loss.backward()
    optimizer.step()
    t.set_description(f"Loss: {loss.item():.6f}")
    loss_list.append(loss.item())
    
    

head.weight torch.Size([192, 3072]) True
head.bias torch.Size([192]) True
model.conv2.weight torch.Size([48, 48, 1, 1, 1]) True
model.conv2.bias torch.Size([48]) True
model.decoder.conv1.weight torch.Size([1024, 48, 3, 3, 3]) True
model.decoder.conv1.bias torch.Size([1024]) True
model.decoder.middle.0.residual.0.gamma torch.Size([1024, 1, 1, 1]) True
model.decoder.middle.0.residual.2.weight torch.Size([1024, 1024, 3, 3, 3]) True
model.decoder.middle.0.residual.2.bias torch.Size([1024]) True
model.decoder.middle.0.residual.3.gamma torch.Size([1024, 1, 1, 1]) True
model.decoder.middle.0.residual.6.weight torch.Size([1024, 1024, 3, 3, 3]) True
model.decoder.middle.0.residual.6.bias torch.Size([1024]) True
model.decoder.middle.1.norm.gamma torch.Size([1024, 1, 1]) True
model.decoder.middle.1.to_qkv.weight torch.Size([3072, 1024, 1, 1]) True
model.decoder.middle.1.to_qkv.bias torch.Size([3072]) True
model.decoder.middle.1.proj.weight torch.Size([1024, 1024, 1, 1]) True
model.decoder.middle.

  0%|                                                                                        | 0/100 [00:00<?, ?it/s]

torch.Size([1, 2400, 3072])
torch.Size([1, 2400, 192])
torch.Size([1, 48, 12, 20, 40])


Loss: 1.087572:   1%|▋                                                               | 1/100 [00:02<04:27,  2.70s/it]

torch.Size([1, 2400, 3072])
torch.Size([1, 2400, 192])
torch.Size([1, 48, 12, 20, 40])


...

torch.Size([1, 2400, 3072])
torch.Size([1, 2400, 192])
torch.Size([1, 48, 12, 20, 40])


Loss: 1.005781:   9%|█████▊                                                          | 9/100 [00:20<03:24,  2.25s/it]

torch.Size([1, 2400, 3072])
torch.Size([1, 2400, 192])
torch.Size([1, 48, 12, 20, 40])


Loss: 1.004313:  10%|██████▎                                                        | 10/100 [00:22<03:22,  2.24s/it]

torch.Size([1, 2400, 3072])
torch.Size([1, 2400, 192])
torch.Size([1, 48, 12, 20, 40])
